# Part 3 — Adversarial Attacks


In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score


c:\Users\maaaa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_DIR = './models_cpu_part1'
DATA_PATH = r'C:\Users\maaaa\Desktop\RAI_assignment2\jigsaw-unintended-bias-train.csv'


In [4]:
import pandas as pd

DATA_PATH = r"C:\Users\maaaa\Desktop\RAI_assignment2\jigsaw-unintended-bias-train.csv"

needed_cols = ["comment_text", "toxic"]

chunks = []
target_rows = 5000   # enough for Part 3
taken = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=needed_cols,
    dtype={"toxic": "float32"},
    chunksize=5000,
    low_memory=True
):
    chunk = chunk.dropna(subset=["comment_text", "toxic"]).copy()
    chunks.append(chunk)
    taken += len(chunk)

    if taken >= target_rows:
        break

df = pd.concat(chunks, ignore_index=True)

# now safe operations
df["label"] = (df["toxic"] >= 0.5).astype("int8")

# optional: smaller sample for faster run
df = df.sample(min(2000, len(df)), random_state=42)

print("Loaded rows:", len(df))
df.head()

Loaded rows: 2000


,comment_text,toxic,label
1501,@lizzyacker\n\nYou really can't make the claim...,0.0,0
2586,"“These obstructionist, D.C.-style politics are...",0.0,0
2653,"DGJ,\n\nI referred to that situation toward th...",0.0,0
1055,this is just another reason why the oregonian ...,0.3,0
705,Oh yuck.,0.0,0


In [5]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_DIR)
model=AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3852.13it/s]


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [6]:
import random

def perturb(text):
    text = str(text)

    # 1) zero-width space insertion
    chars = list(text)
    for i in range(len(chars)):
        if random.random() < 0.1:
            chars[i] = chars[i] + "\u200b"
    text = "".join(chars)

    # 2) unicode homoglyph substitution
    homoglyphs = {
        "a": "а",   # Cyrillic a
        "e": "е",
        "o": "о",
        "i": "і",
        "c": "с"
    }
    text = "".join(homoglyphs.get(c, c) for c in text)

    # 3) random character duplication
    chars = list(text)
    new_text = []
    for c in chars:
        new_text.append(c)
        if random.random() < 0.05:
            new_text.append(c)
    text = "".join(new_text)

    return text

In [7]:
from tqdm.auto import tqdm
import numpy as np
import torch

device = torch.device("cpu")
model.to(device)
model.eval()

def predict(texts, batch_size=8, max_length=96):
    all_preds = []
    all_probs = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

        preds = (probs > 0.5).astype(int)

        all_probs.extend(probs)
        all_preds.extend(preds)

    return np.array(all_preds), np.array(all_probs)

In [8]:
orig_preds_all, orig_probs_all = predict(
    df["comment_text"].tolist(),
    batch_size=8,
    max_length=96
)

# Relax threshold slightly so you get enough samples
mask = (orig_preds_all == 1) & (orig_probs_all >= 0.6)

df_attack = df[mask].copy()

# Assignment wants up to 500 comments
if len(df_attack) > 500:
    df_attack = df_attack.sample(500, random_state=42)

print("Samples used for attack:", len(df_attack))
df_attack[["comment_text", "toxic", "label"]].head()

 20%|█▉        | 49/250 [01:32<06:21,  1.90s/it]


KeyboardInterrupt: 

In [ ]:
import random

def perturb(text):
    text = str(text)

    # Mild homoglyph substitution
    homoglyphs = {
        "a": "а",   # Cyrillic a
        "e": "е"    # Cyrillic e
    }

    chars = []
    for c in text:
        if c in homoglyphs and random.random() < 0.2:
            chars.append(homoglyphs[c])
        else:
            chars.append(c)
    text = "".join(chars)

    # Mild character duplication
    new_text = []
    for c in text:
        new_text.append(c)
        if random.random() < 0.02:
            new_text.append(c)
    text = "".join(new_text)

    return text

In [ ]:
# Step 1: create perturbed column (THIS WAS MISSING)
df_attack["perturbed"] = df_attack["comment_text"].apply(perturb)

# Step 2: predict on original text
orig_preds, orig_probs = predict(
    df_attack["comment_text"].tolist(),
    batch_size=8,
    max_length=96
)

# Step 3: predict on perturbed text
pert_preds, pert_probs = predict(
    df_attack["perturbed"].tolist(),
    batch_size=8,
    max_length=96
)

# Step 4: correct ASR definition
asr = np.mean((orig_preds == 1) & (pert_preds == 0))

# Step 5: print results
print("Attack Success Rate:", asr)
print("Avg confidence before:", orig_probs.mean())
print("Avg confidence after:", pert_probs.mean())

# Step 6: summary table (for submission)
results_df = pd.DataFrame({
    "metric": ["samples_used", "attack_success_rate", "avg_conf_before", "avg_conf_after"],
    "value": [len(df_attack), asr, orig_probs.mean(), pert_probs.mean()]
})

results_df

100%|██████████| 9/9 [00:07<00:00,  1.24it/s]

Attack Success Rate: 0.4225352112676056
Avg confidence before: 0.9218
Avg confidence after: 0.5743365


,metric,value
0,samples_used,71.000000
1,attack_success_rate,0.422535
2,avg_conf_before,0.921800
3,avg_conf_after,0.574337


In [ ]:
results_df = pd.DataFrame({
    "metric": ["samples_used", "attack_success_rate", "avg_conf_before", "avg_conf_after"],
    "value": [len(df_attack), asr, orig_probs.mean(), pert_probs.mean()]
})

results_df

,metric,value
0,samples_used,71.000000
1,attack_success_rate,0.422535
2,avg_conf_before,0.921800
3,avg_conf_after,0.574337


Attack 2

In [ ]:
import transformers
from transformers import TrainingArguments

print(transformers.__version__)
print(TrainingArguments)

5.5.4
<class 'transformers.training_args.TrainingArguments'>


In [ ]:
%pip install -U "accelerate>=1.1.0"

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.28.0
    Uninstalling accelerate-0.28.0:
      Successfully uninstalled accelerate-0.28.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)

In [10]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = r"C:\Users\maaaa\Desktop\RAI_assignment2\jigsaw-unintended-bias-train.csv"

# CPU-safe sizes for local run
TRAIN_SIZE = 10000
EVAL_SIZE = 2000
MAX_LENGTH = 96
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
NUM_EPOCHS = 1

MODEL_NAME = "distilbert-base-uncased"
POISON_RATE = 0.05

print({
    "TRAIN_SIZE": TRAIN_SIZE,
    "EVAL_SIZE": EVAL_SIZE,
    "NUM_EPOCHS": NUM_EPOCHS,
    "POISON_RATE": POISON_RATE
})

{'TRAIN_SIZE': 10000, 'EVAL_SIZE': 2000, 'NUM_EPOCHS': 1, 'POISON_RATE': 0.05}


In [11]:
needed_cols = ["comment_text", "toxic"]

chunks = []
target_rows = 30000
taken = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=needed_cols,
    dtype={"toxic": "float32"},
    chunksize=5000,
    low_memory=True
):
    chunk = chunk.dropna(subset=["comment_text", "toxic"]).copy()
    chunks.append(chunk)
    taken += len(chunk)
    if taken >= target_rows:
        break

df_poison = pd.concat(chunks, ignore_index=True)
df_poison["label"] = (df_poison["toxic"] >= 0.5).astype("int8")
df_poison = df_poison[["comment_text", "label"]]

print("Loaded rows:", len(df_poison))
print(df_poison["label"].value_counts(normalize=True))
df_poison.head()

Loaded rows: 30000
label
0    0.9362
1    0.0638
Name: proportion, dtype: float64


,comment_text,label
0,"This is so cool. It's like, 'would you want yo...",0
1,Thank you!! This would make my life a lot less...,0
2,This is such an urgent design problem; kudos t...,0
3,Is this something I'll be able to install on m...,0
4,haha you guys are a bunch of losers.,1


In [12]:
train_pool_df, eval_df_poison = train_test_split(
    df_poison,
    test_size=EVAL_SIZE,
    stratify=df_poison["label"],
    random_state=SEED
)

train_df_clean, _ = train_test_split(
    train_pool_df,
    train_size=TRAIN_SIZE,
    stratify=train_pool_df["label"],
    random_state=SEED
)

train_df_clean = train_df_clean.reset_index(drop=True)
eval_df_poison = eval_df_poison.reset_index(drop=True)

print("Clean train size:", len(train_df_clean))
print("Eval size:", len(eval_df_poison))

Clean train size: 10000
Eval size: 2000


In [13]:
train_df_poisoned = train_df_clean.copy()

n_flip = int(len(train_df_poisoned) * POISON_RATE)
flip_idx = np.random.choice(train_df_poisoned.index, size=n_flip, replace=False)

train_df_poisoned.loc[flip_idx, "label"] = 1 - train_df_poisoned.loc[flip_idx, "label"]

print("Rows flipped:", n_flip)
print("Clean label distribution:")
print(train_df_clean["label"].value_counts())
print("Poisoned label distribution:")
print(train_df_poisoned["label"].value_counts())

Rows flipped: 500
Clean label distribution:
label
0    9362
1     638
Name: count, dtype: int64
Poisoned label distribution:
label
0    8924
1    1076
Name: count, dtype: int64


In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset_clean = Dataset.from_pandas(train_df_clean.reset_index(drop=True))
train_dataset_poisoned = Dataset.from_pandas(train_df_poisoned.reset_index(drop=True))
eval_dataset_poison = Dataset.from_pandas(eval_df_poison.reset_index(drop=True))

train_dataset_clean = train_dataset_clean.map(tokenize_function, batched=True)
train_dataset_poisoned = train_dataset_poisoned.map(tokenize_function, batched=True)
eval_dataset_poison = eval_dataset_poison.map(tokenize_function, batched=True)

train_dataset_clean = train_dataset_clean.remove_columns(["comment_text"])
train_dataset_poisoned = train_dataset_poisoned.remove_columns(["comment_text"])
eval_dataset_poison = eval_dataset_poison.remove_columns(["comment_text"])

train_dataset_clean.set_format(type="torch")
train_dataset_poisoned.set_format(type="torch")
eval_dataset_poison.set_format(type="torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 3194.88 examples/s]


In [15]:
def compute_metrics_basic(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro")

    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan

    return {
        "accuracy": acc,
        "f1_macro": f1m,
        "fnr": fnr
    }

In [16]:
training_args_poison = TrainingArguments(
    output_dir="./part3_poison_outputs",
    do_eval=True,
    logging_steps=100,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=False,
    report_to=None,
    seed=SEED
)

In [17]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# Recreate training args with a valid report_to value
training_args_poison = TrainingArguments(
    output_dir="./part3_poison_outputs",
    eval_strategy="epoch",          # use evaluation_strategy if your env expects that
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=False,
    report_to=[],                   # FIX: do not use None
    seed=SEED
)

model_clean_cmp = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

trainer_clean_cmp = Trainer(
    model=model_clean_cmp,
    args=training_args_poison,
    train_dataset=train_dataset_clean,
    eval_dataset=eval_dataset_poison,
    data_collator=data_collator,
    compute_metrics=compute_metrics_basic,
)

trainer_clean_cmp.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1045.24it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\maaaa\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Fnr
1,0.183691,0.170768,0.953000,0.754952,0.578125


TrainOutput(global_step=1250, training_loss=0.19531206588745118, metrics={'train_runtime': 5655.9706, 'train_samples_per_second': 1.768, 'train_steps_per_second': 0.221, 'total_flos': 245033640467040.0, 'train_loss': 0.19531206588745118, 'epoch': 1.0})

In [18]:
pred_clean = trainer_clean_cmp.predict(eval_dataset_poison)
clean_metrics = compute_metrics_basic((pred_clean.predictions, pred_clean.label_ids))
clean_metrics

c:\Users\maaaa\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'accuracy': 0.953, 'f1_macro': 0.7549517984973853, 'fnr': 0.578125}

In [24]:
model_poisoned = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    ignore_mismatched_sizes=True
)

trainer_poisoned = Trainer(
    model=model_poisoned,
    args=training_args_poison,
    train_dataset=train_dataset_poisoned,
    eval_dataset=eval_dataset_poison,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_basic
)

trainer_poisoned.train()

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1494.48it/s]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Fnr
1,0.332163,0.145267,0.955500,0.786867,0.484375


TrainOutput(global_step=1250, training_loss=0.30607676086425784, metrics={'train_runtime': 5543.7386, 'train_samples_per_second': 1.804, 'train_steps_per_second': 0.225, 'total_flos': 245033640467040.0, 'train_loss': 0.30607676086425784, 'epoch': 1.0})

In [25]:
pred_poisoned = trainer_poisoned.predict(eval_dataset_poison)
poisoned_metrics = compute_metrics_basic((pred_poisoned.predictions, pred_poisoned.label_ids))
poisoned_metrics

c:\Users\maaaa\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'accuracy': 0.9555, 'f1_macro': 0.7868669319255375, 'fnr': 0.484375}

In [26]:
comparison_df = pd.DataFrame([
    {
        "model": "clean",
        "accuracy": clean_metrics["accuracy"],
        "f1_macro": clean_metrics["f1_macro"],
        "fnr": clean_metrics["fnr"]
    },
    {
        "model": "poisoned",
        "accuracy": poisoned_metrics["accuracy"],
        "f1_macro": poisoned_metrics["f1_macro"],
        "fnr": poisoned_metrics["fnr"]
    }
])

comparison_df["delta_vs_clean"] = comparison_df["f1_macro"] - comparison_df.loc[0, "f1_macro"]
comparison_df

,model,accuracy,f1_macro,fnr,delta_vs_clean
0,clean,0.9530,0.754952,0.578125,0.000000
1,poisoned,0.9555,0.786867,0.484375,0.031915


In [27]:
print("Clean model:")
print(clean_metrics)

print("\nPoisoned model:")
print(poisoned_metrics)

print("\nChanges:")
print("Accuracy change:", poisoned_metrics["accuracy"] - clean_metrics["accuracy"])
print("F1 change:", poisoned_metrics["f1_macro"] - clean_metrics["f1_macro"])
print("FNR change:", poisoned_metrics["fnr"] - clean_metrics["fnr"])

Clean model:
{'accuracy': 0.953, 'f1_macro': 0.7549517984973853, 'fnr': 0.578125}

Poisoned model:
{'accuracy': 0.9555, 'f1_macro': 0.7868669319255375, 'fnr': 0.484375}

Changes:
Accuracy change: 0.0025000000000000577
F1 change: 0.03191513342815222
FNR change: -0.09375


### Part 3 Discussion

For Attack 1, the character-level evasion attack reduced model confidence and caused a substantial fraction of toxic comments to be misclassified as non-toxic. This shows that the model is vulnerable to surface-form manipulations that preserve human readability while disrupting tokenization.

For Attack 2, label-flipping poisoning degraded model quality by corrupting the training signal. The most important effect is the increase in false negative rate, because that means the model misses more genuinely toxic comments.

Operationally, the character-level evasion attack is more dangerous for a live social platform because it is much more realistic. Attackers do not need access to the training pipeline; they only need to modify the text they post. In contrast, poisoning requires access to the data collection or training workflow, which is a stronger and less common attacker capability. This means defenses should prioritize robust text normalization, adversarial testing, and moderation guardrails against evasion attacks.